### Source Tables:
- `_exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3address` — patient addresses

### Strategy:
- Filter to TypeCode = 'Home', Active = 1, IsCurrent = 1 for current patient addresses
- Deduplicate to one address per GUID (most recently touched)
- Map fields: Line1 → address_1, Line2 → address_2, City → city, CountryDvsnCode → state, PostalCode → zip, County → county
- No country concept mapping (SCM does not have a country field — default to 0)

### Notes:
- ParentGUID links back to cv3client.GUID for patient association
- ~34M current home addresses with populated fields
- Multiple addresses per patient exist; we take current + most recent

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.location;

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_scm.location;


In [0]:
%sql
DELETE FROM _exponent.omop_silver.location
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_location
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver AS
SELECT
  LOWER(TRIM(addr.Line1)) AS address_1,
  LOWER(TRIM(addr.Line2)) AS address_2,
  LOWER(TRIM(addr.City)) AS city,
  LOWER(TRIM(addr.CountryDvsnCode)) AS state,
  LOWER(TRIM(addr.PostalCode)) AS zip,
  LOWER(TRIM(addr.County)) AS county,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3address', 'GUID', CAST(addr.GUID AS STRING)) AS location_source_value,
  0 AS country_concept_id,
  NULL AS country_source_value,
  NULL AS latitude,
  NULL AS longitude,
  'allscripts_scm' AS source_system
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3address addr
WHERE 
  Active = 1
  AND TypeCode = 'Home'
  AND Line1 IS NOT NULL


In [0]:
%sql
MERGE INTO _exponent.omop_silver.location AS target
USING silver AS source
ON target.location_source_value = source.location_source_value

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.country_source_value   <=> source.country_source_value
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.source_system          <=> source.source_system
) THEN UPDATE SET
  target.address_1              = source.address_1,
  target.address_2              = source.address_2,
  target.city                   = source.city,
  target.state                  = source.state,
  target.zip                    = source.zip,
  target.county                 = source.county,
  target.country_concept_id     = source.country_concept_id,
  target.country_source_value   = source.country_source_value,
  target.latitude               = source.latitude,
  target.longitude              = source.longitude,
  target.source_system          = source.source_system,
  target.last_mod_tsp           = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  country_source_value,
  latitude,
  longitude,
  source_system,
  last_mod_tsp
) VALUES (
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.country_source_value,
  source.latitude,
  source.longitude,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_location (
    source_system,
    location_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    silver_location.source_system,
    silver_location.location_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(silver_location.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        location_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.location
    WHERE source_system = 'allscripts_scm'
      AND location_source_value IS NOT NULL
) AS silver_location
LEFT ANTI JOIN _exponent.omop_mapping.source_to_location AS existing_location
  ON silver_location.location_source_value = existing_location.location_source_value;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  source_to_location.location_id,
  location.address_1,
  location.address_2,
  location.city,
  location.state,
  location.zip,
  location.county,
  location.country_concept_id,
  location.latitude,
  location.longitude,
  location.location_source_value,
  location.last_mod_tsp
FROM _exponent.omop_silver.location AS location
JOIN _exponent.omop_mapping.source_to_location AS source_to_location
  ON location.location_source_value = source_to_location.location_source_value
 AND source_to_location.active_flag = TRUE
WHERE location.source_system = 'allscripts_scm'

In [0]:
%sql
MERGE INTO _exponent.omop_scm.location AS target
USING gold AS source
ON target.location_id = source.location_id

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.location_source_value  <=> source.location_source_value
) THEN UPDATE SET
  target.address_1               = source.address_1,
  target.address_2               = source.address_2,
  target.city                    = source.city,
  target.state                   = source.state,
  target.zip                     = source.zip,
  target.county                  = source.county,
  target.country_concept_id      = source.country_concept_id,
  target.latitude                = source.latitude,
  target.longitude               = source.longitude,
  target.location_source_value   = source.location_source_value

WHEN NOT MATCHED THEN INSERT (
  location_id,
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  latitude,
  longitude
) VALUES (
  source.location_id,
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.latitude,
  source.longitude
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.location AS target
USING gold AS source
ON target.location_id = source.location_id

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.location_source_value  <=> source.location_source_value
) THEN UPDATE SET
  target.address_1               = source.address_1,
  target.address_2               = source.address_2,
  target.city                    = source.city,
  target.state                   = source.state,
  target.zip                     = source.zip,
  target.county                  = source.county,
  target.country_concept_id      = source.country_concept_id,
  target.latitude                = source.latitude,
  target.longitude               = source.longitude,
  target.location_source_value   = source.location_source_value

WHEN NOT MATCHED THEN INSERT (
  location_id,
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  latitude,
  longitude
) VALUES (
  source.location_id,
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.latitude,
  source.longitude
);